<a href="https://colab.research.google.com/github/f8th/langchain-practice/blob/main/langchain_chrome_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["OPENAI_API_KEY"]

In [32]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community langchain-chroma faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.8 MB/s eta 0:00:00


In [33]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

In [34]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [35]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [36]:
vector_store = Chroma(
    collection_name='players',
    embedding_function=OpenAIEmbeddings(),
    persist_directory='./chroma_db'
)

In [45]:
db = FAISS.from_documents(docs, OpenAIEmbeddings())

In [46]:
db.add_documents(docs)

['bea9057b-41f3-4676-a50d-c1ffc30fa0ea',
 'fe8eae2e-7366-439f-9ae0-006ca6e80657',
 '3f91d735-3762-4aa0-842b-e81e40f2a053',
 'cd5a40bc-3d4d-4572-a5c3-97e85b25b4bc',
 '076101ee-7111-4bc0-900a-2118c276fed0']

In [12]:
vector_store.add_documents(docs)

['460c6940-ae23-4410-a83c-1d85540d1b9e',
 'c0e05c3f-81ec-49bb-ae26-40a897ccb109',
 'f8bb7f47-d612-4c15-b25e-5f3f06afe0c2',
 '59596322-a6d1-4ec1-8066-ec7a2bfeb91c',
 '84a23c77-d710-4b77-84b4-35ad5baa2a4a']

In [52]:
db.get_by_ids(
    ['bea9057b-41f3-4676-a50d-c1ffc30fa0ea',
     'fe8eae2e-7366-439f-9ae0-006ca6e80657'
    ]
)

[Document(id='bea9057b-41f3-4676-a50d-c1ffc30fa0ea', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
 Document(id='fe8eae2e-7366-439f-9ae0-006ca6e80657', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [54]:
db.save_local("my_faiss_index")

In [13]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['460c6940-ae23-4410-a83c-1d85540d1b9e',
  'c0e05c3f-81ec-49bb-ae26-40a897ccb109',
  'f8bb7f47-d612-4c15-b25e-5f3f06afe0c2',
  '59596322-a6d1-4ec1-8066-ec7a2bfeb91c',
  '84a23c77-d710-4b77-84b4-35ad5baa2a4a'],
 'embeddings': array([[-2.07734108e-03, -2.16356432e-03,  2.67739072e-02, ...,
         -1.70658119e-02, -3.68524459e-03,  1.35402400e-02],
        [-2.70681921e-03, -5.47363961e-05,  2.82030534e-02, ...,
         -1.50933377e-02,  5.93815418e-03, -1.16601437e-02],
        [ 7.83727213e-04, -4.76402277e-03,  1.23708826e-02, ...,
         -1.72387529e-02,  7.74397107e-04,  2.93532619e-03],
        [-2.71391869e-02,  8.87096301e-03,  2.69362777e-02, ...,
         -2.58583203e-02,  9.02314577e-03, -1.99992992e-02],
        [-1.81433018e-02,  1.27527108e-02,  3.47932205e-02, ...,
         -3.03654689e-02, -5.85236121e-03,  5.21374308e-03]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style 

In [14]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler',
    k=2
)

[(Document(id='59596322-a6d1-4ec1-8066-ec7a2bfeb91c', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.34601330757141113),
 (Document(id='84a23c77-d710-4b77-84b4-35ad5baa2a4a', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.4080040454864502)]

In [15]:
update_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)


In [17]:
vector_store.update_document(
    document_id='460c6940-ae23-4410-a83c-1d85540d1b9e',
    document=update_doc1
)

In [20]:
vector_store.get(
    ids='460c6940-ae23-4410-a83c-1d85540d1b9e'
)

{'ids': ['460c6940-ae23-4410-a83c-1d85540d1b9e'],
 'embeddings': None,
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket."],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'team': 'Royal Challengers Bangalore'}]}

In [22]:
vector_store.delete(
    ids='460c6940-ae23-4410-a83c-1d85540d1b9e'
)